## Consolidação e Checagem de Qualidade dos Dados

Após finalizar a etapa inicial de entendimento dos dados com recorte do ano de 2025, iremos para a próxima fase que será:

- Realizar a junção de todos os nossos dados, desde o ano 2000 até 2025, transformando em um único arquivo parquet. A partir dele que daremos seguimento a parte de análise.
- Outro passo, é verificar a qualidade desses dados que temos, será que estão duplicados, se possui valores ausentes ou valores inconsistentes... São verificações que faremos nessa etapa.
- Por fim, após finalizado os dois passos anteriores, salvaremos nosso arquivo na pasta 'data/processed'.

### 1. Importação das Bibliotecas

In [2]:
from pathlib import Path
import pandas as pd

RAW_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')

### 2. Concatenando os Dados e Formando Dataset Histórico

Agora, iremos pegar todos os dados que estão no formato parquet da pasta 'data/raw' e vamos fazer a junção deles, transformando em um único DataFrame. Com isso, vamos ter completamente o histórico desde o ano 2000 até 2025.

In [3]:
files = sorted(RAW_PATH.glob('curva_carga_*.parquet'))

print(f'{len(files)} arquivos encontrados.')

electric_data = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
electric_data.head()


26 arquivos encontrados.


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
0,N,NORTE,2000-01-01 00:00:00,2373.69999999
1,NE,NORDESTE,2000-01-01 00:00:00,5340.20000000
2,S,SUL,2000-01-01 00:00:00,5777.00000000
3,SE,SUDESTE,2000-01-01 00:00:00,21182.99999999
4,N,NORTE,2000-01-01 01:00:00,2331.60000000


#### 2.1. Verificando Data Inicial e Final

In [4]:
print(f"Data Inicial: {electric_data['din_instante'].min()}")
print(f"Data Final: {electric_data['din_instante'].max()}")

Data Inicial: 2000-01-01 00:00:00
Data Final: 2025-12-31 23:00:00


#### 2.2. Dimensão do Dataset

In [5]:
print(f'O dataset final possui um total de {electric_data.shape[0]} linhas e {electric_data.shape[1]} colunas.')

O dataset final possui um total de 911608 linhas e 4 colunas.


### 3. Informações Gerais

Antes de seguir com as próximas etapas, vamos visualizar as informações gerais desse dataset e verificar se existe algum valor nulo e se os tipos de dados estão corretos.

In [6]:
electric_data.info(memory_usage='deep')


<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911608 non-null  object        
dtypes: datetime64[ns](1), object(1), str(2)
memory usage: 80.4 MB


Temos um problema !

Ao verificar o tipo de cada coluna, o valor de carga está vindo com o tipo object, quando na verdade deveria ser float, por ser um valor decimal. Será explorado melhor esta variável para podermos identificar alguma inconsistência.

#### 3.1. Tipos de Dado das Cargas por Ano

Inicialmente, irei verificar qual o tipo de dado que a coluna de carga recebeu nos arquivos de cada ano.

In [7]:
for file in files:
    year_data = pd.read_parquet(file, columns=['val_cargaenergiahomwmed'])
    year = file.stem.split('_')[-1]
    column_type = year_data['val_cargaenergiahomwmed'].dtype

    print(f'{year}: {column_type}')


2000: str
2001: str
2002: str
2003: str
2004: str
2005: str
2006: str
2007: str
2008: str
2009: str
2010: str
2011: str
2012: str
2013: str
2014: str
2015: str
2016: str
2017: str
2018: str
2019: str
2020: str
2021: str
2022: str
2023: str
2024: str
2025: float64


In [8]:
empty_load = electric_data[electric_data['val_cargaenergiahomwmed'] == '']

print(f'Quantidade de valores vazios: {empty_load.shape[0]}')

empty_load.head()

Quantidade de valores vazios: 259


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
487912,N,NORTE,2013-12-01 00:00:00,
487913,NE,NORDESTE,2013-12-01 00:00:00,
487914,S,SUL,2013-12-01 00:00:00,
487915,SE,SUDESTE,2013-12-01 00:00:00,
487916,N,NORTE,2013-12-01 01:00:00,


Nos arquivos de 2000 a 2024, a coluna de carga está armazenada como texto, enquanto no arquivo de 2025 ela está com o tipo `float64`. Como realizei a concatenação de arquivos com tipos diferentes, o Pandas representou a coluna consolidada como `object`.

Além da diferença entre os tipos, foram encontrados 259 registros contendo strings vazias (`''`). Na próxima etapa, vamos verificar como esses valores se comportam durante a conversão da coluna.

#### 3.2. Conversão da Coluna de Carga

In [9]:
after_conversion = pd.to_numeric(electric_data['val_cargaenergiahomwmed'], errors='coerce')

print(f"Quantidade de valores ausentes antes da conversão: {electric_data['val_cargaenergiahomwmed'].isna().sum()}")
print(f"Quantidade de valores ausentes após a conversão: {after_conversion.isna().sum()}")


Quantidade de valores ausentes antes da conversão: 0
Quantidade de valores ausentes após a conversão: 259


Na primeira verificação com o `info()`, a coluna de carga aparentemente não possuía valores nulos. Isso aconteceu porque as strings vazias eram consideradas textos válidos pelo Pandas.

Depois da conversão, os 259 campos vazios passaram a ser representados como `NaN`, permitindo que a coluna fosse armazenada corretamente como `float64`.

In [10]:
electric_data['val_cargaenergiahomwmed'] = after_conversion

In [11]:
electric_data.info(memory_usage='deep')


<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911349 non-null  float64       
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 34.1 MB


### 4. Verificação dos Valores Ausentes

Agora, vamos contar os valores ausentes em cada coluna. Em seguida, criaremos uma cópia do dataset contendo apenas os registros nulos para analisar em quais anos eles aparecem com maior frequência.

In [12]:
electric_data.isna().sum()

id_subsistema                0
nom_subsistema               0
din_instante                 0
val_cargaenergiahomwmed    259
dtype: int64

In [13]:
missing_data = electric_data[electric_data['val_cargaenergiahomwmed'].isna()].copy()
missing_data.head()

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
487912,N,NORTE,2013-12-01 00:00:00,NaN
487913,NE,NORDESTE,2013-12-01 00:00:00,NaN
487914,S,SUL,2013-12-01 00:00:00,NaN
487915,SE,SUDESTE,2013-12-01 00:00:00,NaN
487916,N,NORTE,2013-12-01 01:00:00,NaN


In [14]:
missing_data['ano'] = missing_data['din_instante'].dt.year
missing_data['ano'].value_counts().sort_index()

ano
2013    96
2014    76
2015    76
2016     4
2017     4
2018     3
Name: count, dtype: int64

Os casos estão concentrados entre 2013 e 2018, com as maiores quantidades em 2013, 2014 e 2015. Para entender melhor essa distribuição, vamos detalhar os valores ausentes por data e subsistema.

In [15]:
missing_data['data'] = missing_data['din_instante'].dt.date

pd.crosstab(missing_data['data'], missing_data['id_subsistema'])

id_subsistema,N,NE,S,SE
data,,,,
2013-12-01,24,24,24,24
2014-02-01,0,24,24,24
2014-10-19,1,1,1,1
2015-04-09,0,24,24,24
2015-10-18,1,1,1,1
2016-10-16,1,1,1,1
2017-10-15,1,1,1,1
2018-11-04,1,1,0,1


In [16]:
missing_data.groupby(['data', 'id_subsistema'])['din_instante'].agg(['count', 'nunique', 'min', 'max'])

count  nunique        min                 max
data       id_subsistema                                               
2013-12-01 N                 24       24 2013-12-01 2013-12-01 23:00:00
           NE                24       24 2013-12-01 2013-12-01 23:00:00
           S                 24       24 2013-12-01 2013-12-01 23:00:00
           SE                24       24 2013-12-01 2013-12-01 23:00:00
2014-02-01 NE                24       24 2014-02-01 2014-02-01 23:00:00
           S                 24       24 2014-02-01 2014-02-01 23:00:00
           SE                24       24 2014-02-01 2014-02-01 23:00:00
2014-10-19 N                  1        1 2014-10-19 2014-10-19 00:00:00
           NE                 1        1 2014-10-19 2014-10-19 00:00:00
           S                  1        1 2014-10-19 2014-10-19 00:00:00
           SE                 1        1 2014-10-19 2014-10-19 00:00:00
2015-04-09 NE                24       24 2015-04-09 2015-04-09 23:00:00
           S                 24       24 2015-04-09 2015-04-09 23:00:00
           SE                24       24 2015-04-09 2015-04-09 23:00:00
2015-10-18 N                  1        1 2015-10-18 2015-10-18 00:00:00
           NE                 1        1 2015-10-18 2015-10-18 00:00:00
           S                  1        1 2015-10-18 2015-10-18 00:00:00
           SE                 1        1 2015-10-18 2015-10-18 00:00:00
2016-10-16 N                  1        1 2016-10-16 2016-10-16 00:00:00
           NE                 1        1 2016-10-16 2016-10-16 00:00:00
           S                  1        1 2016-10-16 2016-10-16 00:00:00
           SE                 1        1 2016-10-16 2016-10-16 00:00:00
2017-10-15 N                  1        1 2017-10-15 2017-10-15 00:00:00
           NE                 1        1 2017-10-15 2017-10-15 00:00:00
           S                  1        1 2017-10-15 2017-10-15 00:00:00
           SE                 1        1 2017-10-15 2017-10-15 00:00:00
2018-11-04 N                  1        1 2018-11-04 2018-11-04 00:00:00
           NE                 1        1 2018-11-04 2018-11-04 00:00:00
           SE                 1        1 2018-11-04 2018-11-04 00:00:00

O que foi possível concluir com essas verificações?

Nas colunas `min` e `max`, quando o Pandas apresenta somente a data, sem mostrar o horário, significa que aquele registro corresponde às 00:00:00. O horário é omitido da exibição quando corresponde exatamente à meia-noite.

Em 01/12/2013, cada subsistema possui 24 registros ausentes e 24 horários distintos. O primeiro horário é 00:00 e o último é 23:00, confirmando que a carga está ausente em todas as horas do dia.

Em 01/02/2014 e 09/04/2015, o mesmo comportamento aparece nos subsistemas Nordeste, Sul e Sudeste: são 24 horários distintos, das 00:00 às 23:00. O Norte não aparece entre os valores nulos, portanto ainda será verificado se seus registros existem nessas datas.

Em 19/10/2014, 18/10/2015, 16/10/2016 e 15/10/2017, cada subsistema possui somente um valor ausente. Como `count` e `nunique` são iguais a 1 e os horários mínimo e máximo correspondem a 00:00, podemos confirmar que a ausência aconteceu apenas à meia-noite.

Em 04/11/2018, também existe uma ausência às 00:00, mas somente nos subsistemas Norte, Nordeste e Sudeste. O Sul não aparece entre os registros nulos e, por isso, seu valor nesse mesmo horário será investigado separadamente.

Até aqui, identificamos quando as ausências aconteceram, mas ainda não investigamos suas possíveis causas.

### 5. Análise dos Casos Identificados

Nesta etapa, vamos analisar duas situações identificadas anteriormente:

- o subsistema Sul não aparece entre os valores nulos de 04/11/2018 às 00:00, enquanto os outros três subsistemas apresentam carga ausente;
- o subsistema Norte não aparece entre os valores nulos de 01/02/2014 e 09/04/2015, enquanto os outros três subsistemas possuem 24 valores ausentes nessas datas.

#### 5.1. Caso 1 — Subsistema Sul em 04/11/2018

Para esse caso, vamos utilizar todo o conjunto de dados, pois o `missing_data` contém apenas os valores nulos. O fato de o Sul não aparecer não significa que seu registro esteja ausente.

In [17]:
electric_data[electric_data['din_instante'] == pd.Timestamp('2018-11-04 00:00:00')]


,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
660568,N,NORTE,2018-11-04,NaN
660569,NE,NORDESTE,2018-11-04,NaN
660570,S,SUL,2018-11-04,0.0
660571,SE,SUDESTE,2018-11-04,NaN


A consulta mostrou que o registro do subsistema SUL existe, mas sua carga foi registrada como zero. Como zero não é considerado um valor nulo pelo Pandas, esse registro não apareceu na análise anterior. Antes de tomar qualquer tipo de decisão, vamos verificar se existem outros valores iguais ou menores que zero no histórico.

In [18]:
electric_data[electric_data['val_cargaenergiahomwmed'] <= 0]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed
660570,S,SUL,2018-11-04,0.0


A verificação mostrou que existe apenas um valor igual ou inferior a zero em todo o histórico: a carga igual a zero do subsistema Sul em 04/11/2018 às 00:00.

Esse registro ocorre exatamente no mesmo horário em que Norte, Nordeste e Sudeste apresentam valores ausentes. Além disso, uma carga igual a zero indicaria que todo o subsistema ficou sem demanda elétrica durante aquela hora, comportamento inconsistente com esse tipo de série.

Por esses motivos, o valor zero será interpretado como uma carga ausente registrada de forma diferente dos outros subsistemas.

In [19]:
electric_data.loc[electric_data['val_cargaenergiahomwmed'] <= 0, 'val_cargaenergiahomwmed'] = pd.NA

In [20]:
electric_data['val_cargaenergiahomwmed'].isna().sum()

np.int64(260)

Após o tratamento, a quantidade de cargas ausentes passou de 259 para 260. Nenhum registro foi removido do dataset; apenas o valor zero do subsistema Sul foi reclassificado como `NaN`.

#### 5.2. Caso 2 — Subsistema Norte em 01/02/2014 e 09/04/2015

Na análise anterior, o subsistema Norte não apareceu entre os valores nulos de 01/02/2014 e 09/04/2015. Entretanto, isso não significa necessariamente que sua carga esteja preenchida.

Como o `missing_data` contém somente linhas existentes com carga nula, vamos consultar o dataset completo para verificar se existem registros do Norte nessas duas datas.

In [ ]:
dates_to_check = [pd.Timestamp('2014-02-01').date(), pd.Timestamp('2015-04-09').date()]

electric_data[(electric_data['id_subsistema'] == 'N') & (electric_data['din_instante'].dt.date.isin(dates_to_check))]

,id_subsistema,nom_subsistema,din_instante,val_cargaenergiahomwmed


A consulta retornou um DataFrame vazio, confirmando que não existem registros do subsistema Norte em 01/02/2014 e 09/04/2015.

O problema é diferente do encontrado nos outros subsistemas. Nordeste, Sul e Sudeste possuem as 24 linhas de cada data, mas suas cargas estão preenchidas com `NaN`. No Norte, as próprias linhas correspondentes às 24 horas não existem.

Portanto, nessas duas datas, nenhum dos quatro subsistemas possui carga elétrica válida. No Norte existem registros horários ausentes, enquanto nos outros três subsistemas existem registros com valores de carga ausentes.

Esses dados não serão preenchidos durante a consolidação. A decisão de tratamento dependerá da frequência utilizada posteriormente na análise da série temporal.

### 6. Verificação dos Registros Duplicados

Nesta etapa, vamos verificar se existem linhas repetidas no dataset. Também será analisada a combinação entre subsistema e horário, pois cada subsistema deve possuir apenas uma carga registrada em cada instante.

In [22]:
electric_data.duplicated().sum()

np.int64(0)

In [23]:
electric_data.duplicated(subset=['id_subsistema', 'din_instante']).sum()


np.int64(0)

Não foram encontradas linhas completamente duplicadas. Também não existem duplicidades na combinação entre subsistema e horário. Portanto, cada subsistema possui, no máximo, uma carga registrada em cada instante existente na base.

A ausência de duplicidades não garante que a série esteja completa, pois ainda podem existir horários cujas linhas não foram registradas.

### 7. Verificação da Continuidade Temporal

A análise anterior mostrou que algumas linhas podem estar completamente ausentes do dataset e, por isso, não são identificadas pelo `isna()`.

Nesta etapa, vamos verificar se a quantidade de registros é a mesma entre os subsistemas e se existem outros horários ausentes no histórico.

In [24]:
electric_data['id_subsistema'].value_counts().sort_index()

id_subsistema
N     227866
NE    227914
S     227914
SE    227914
Name: count, dtype: int64

Nordeste, Sul e Sudeste possuem 227.914 registros cada, enquanto o Norte possui 227.866.

A diferença é de 48 registros, quantidade equivalente às 24 horas de 01/02/2014 e às 24 horas de 09/04/2015. Esse resultado é compatível com os dois dias sem registros do Norte identificados anteriormente.

Entretanto, quantidades iguais não garantem que as outras séries estejam completas, pois os subsistemas podem possuir horários ausentes em comum. Por isso, será criada uma sequência horária completa para comparação.

In [ ]:
expected_hours = pd.date_range(start=electric_data['din_instante'].min(), end=electric_data['din_instante'].max(), freq='h')

print(f'Quantidade esperada de horários: {len(expected_hours)}')

Quantidade esperada de horários: 227928


In [26]:
missing_hours = {}

for subsystem in sorted(electric_data['id_subsistema'].unique()):
    recorded_hours = electric_data.loc[electric_data['id_subsistema'] == subsystem, 'din_instante']
    missing_hours[subsystem] = expected_hours.difference(recorded_hours)

    print(f'{subsystem}: {len(missing_hours[subsystem])} horários ausentes')


N: 62 horários ausentes
NE: 14 horários ausentes
S: 14 horários ausentes
SE: 14 horários ausentes


In [27]:
date_counts = []

for subsystem, hours in missing_hours.items():
    counts = pd.Series(hours).dt.date.value_counts()
    counts.name = subsystem
    date_counts.append(counts)

missing_by_date = pd.concat(date_counts, axis=1).fillna(0).astype(int).sort_index()

missing_by_date


,N,NE,S,SE
2000-10-08,1,1,1,1
2001-10-14,1,1,1,1
2002-11-03,1,1,1,1
2003-10-19,1,1,1,1
2004-11-02,1,1,1,1
2005-10-16,1,1,1,1
2006-11-05,1,1,1,1
2007-10-14,1,1,1,1
2008-10-19,1,1,1,1
2009-10-18,1,1,1,1


A tabela mostra que existem 14 datas com um horário ausente em comum nos quatro subsistemas. Além delas, o Norte possui 24 horários ausentes em 01/02/2014 e outros 24 em 09/04/2015.

Vamos verificar quais são os horários ausentes simultaneamente nos quatro subsistemas.

In [28]:
common_missing_hours = missing_hours['N']

for subsystem in ['NE', 'S', 'SE']:
    common_missing_hours = common_missing_hours.intersection(missing_hours[subsystem])

common_missing_hours


DatetimeIndex(['2000-10-08', '2001-10-14', '2002-11-03', '2003-10-19',
               '2004-11-02', '2005-10-16', '2006-11-05', '2007-10-14',
               '2008-10-19', '2009-10-18', '2010-10-17', '2011-10-16',
               '2012-10-21', '2013-10-20'],
              dtype='datetime64[ns]', freq=None)

Os 14 horários ausentes em comum ocorrem sempre às 00:00, uma vez por ano entre 2000 e 2013. As datas coincidem com o início oficial do horário de verão, momento em que os relógios eram adiantados em uma hora.

Antes de 2008, as datas eram definidas por decretos anuais. Como exemplos, o [Decreto nº 3.592/2000](https://www.planalto.gov.br/ccivil_03/decreto/d3592.htm) definiu o início em 08/10/2000 e o [Decreto nº 5.223/2004](https://www.planalto.gov.br/ccivil_03/_ato2004-2006/2004/decreto/d5223impressao.htm) definiu o início em 02/11/2004. A partir de 2008, o [Decreto nº 6.558](https://www.planalto.gov.br/ccivil_03/_ato2007-2010/2008/decreto/d6558.htm) estabeleceu o início no terceiro domingo de outubro. O [Ministério de Minas e Energia](https://www.gov.br/mme/pt-br/assuntos/secretarias/secretaria-nacional-energia-eletrica/horario-de-verao) também documenta o funcionamento e as alterações dessa política.

Entre 2000 e 2013, a linha das 00:00 não aparece no dataset nessas datas. Entre 2014 e 2018, a linha passou a existir, mas sem carga válida, como foi observado anteriormente. Em 2018, o Sul registrou zero, valor que já foi convertido para `NaN`.

Como o mesmo padrão ocorre nos quatro subsistemas, inclusive naqueles cujas regiões não adotavam o horário de verão de maneira uniforme, entendemos que a série foi organizada em uma referência temporal comum. Por isso, esses casos serão tratados como efeito do calendário, e não como falha comum de coleta. As linhas ausentes não serão criadas artificialmente.

#### 7.1. Horários Ausentes Apenas no Norte

Depois de separar os horários ausentes em comum, vamos confirmar quais horários aparecem somente no subsistema Norte.

In [29]:
north_only_hours = missing_hours['N'].difference(common_missing_hours)

pd.Series(north_only_hours).dt.date.value_counts().sort_index()


2014-02-01    24
2015-04-09    24
Name: count, dtype: int64

Os 48 horários exclusivos do Norte correspondem exatamente aos dois dias já investigados: 01/02/2014 e 09/04/2015. Portanto, não foram encontrados outros horários ausentes específicos desse subsistema.

Nessas duas datas, as linhas do Norte não existem. Nos outros três subsistemas, as linhas existem, mas todas as cargas estão ausentes. Diferentemente das mudanças de horário de verão, esses dois casos representam dias completos sem informação válida para nenhum subsistema.

### 8. Resumo dos Valores Ausentes

Após converter o zero do Sul para `NaN`, vamos atualizar o conjunto de cargas ausentes e conferir sua distribuição final por data.

In [30]:
final_missing_data = electric_data[electric_data['val_cargaenergiahomwmed'].isna()].copy()
final_missing_data['data'] = final_missing_data['din_instante'].dt.date

final_missing_data['data'].value_counts().sort_index()


data
2013-12-01    96
2014-02-01    72
2014-10-19     4
2015-04-09    72
2015-10-18     4
2016-10-16     4
2017-10-15     4
2018-11-04     4
Name: count, dtype: int64

Os 260 valores de carga ausentes estão distribuídos da seguinte forma:

- 240 pertencem a três dias completos sem carga válida: 01/12/2013, 01/02/2014 e 09/04/2015;
- 20 pertencem às mudanças para o horário de verão entre 2014 e 2018, com quatro subsistemas afetados em cada data.

Além dos valores `NaN`, existem 104 linhas ausentes na sequência horária: 56 relacionadas às 14 mudanças para o horário de verão entre 2000 e 2013 e 48 referentes aos dois dias sem registros do Norte.

Nenhum valor será preenchido nesta etapa. Também não criaremos as linhas que não existem na fonte. Essa decisão preserva os dados observados e permite escolher o tratamento adequado depois que a frequência da análise for definida.

### 9. Organização e Salvamento do Dataset

Antes de salvar, vamos ordenar o dataset por data e subsistema. Em seguida, o arquivo consolidado será armazenado em `data/processed`, mantendo os arquivos originais sem alterações.

In [31]:
electric_data = electric_data.sort_values(['din_instante', 'id_subsistema']).reset_index(drop=True)

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_PATH / 'curva_carga_consolidado.parquet'

electric_data.to_parquet(output_file, index=False)

print(f'Arquivo salvo em: {output_file}')


Arquivo salvo em: ..\data\processed\curva_carga_consolidado.parquet


### 10. Validação Final

Por fim, vamos carregar o arquivo salvo e repetir as verificações principais para garantir que o resultado final está correto.

In [ ]:
processed_data = pd.read_parquet(output_file)

print(f'Dimensão: {processed_data.shape[0]} linhas e {processed_data.shape[1]} colunas')
print('Valores ausentes na carga:', processed_data['val_cargaenergiahomwmed'].isna().sum())
print('Linhas duplicadas:', processed_data.duplicated().sum())
print('Duplicidades de subsistema e horário:', processed_data.duplicated(subset=['id_subsistema', 'din_instante']).sum()
)
print('Valores iguais ou menores que zero:', (processed_data['val_cargaenergiahomwmed'] <= 0).sum())
print(f"Período: {processed_data['din_instante'].min()} até {processed_data['din_instante'].max()}")


Dimensão: 911608 linhas e 4 colunas
Valores ausentes na carga: 260
Linhas duplicadas: 0
Duplicidades de subsistema e horário: 0
Valores iguais ou menores que zero: 0
Período: 2000-01-01 00:00:00 até 2025-12-31 23:00:00


In [33]:
processed_data.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 911608 entries, 0 to 911607
Data columns (total 4 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id_subsistema            911608 non-null  str           
 1   nom_subsistema           911608 non-null  str           
 2   din_instante             911608 non-null  datetime64[ns]
 3   val_cargaenergiahomwmed  911348 non-null  float64       
dtypes: datetime64[ns](1), float64(1), str(2)
memory usage: 34.1 MB


### 11. Conclusão

O histórico de carga elétrica foi consolidado em um único arquivo Parquet com 911.608 registros e quatro colunas, cobrindo o período de 2000 a 2025.

Durante a checagem de qualidade:

- a coluna de carga foi convertida de `object` para `float64`;
- 259 strings vazias foram reconhecidas como valores ausentes;
- um valor igual a zero foi reclassificado como `NaN`;
- não foram encontradas duplicidades;
- foram identificados 260 valores de carga ausentes em linhas existentes;
- foram identificadas 104 linhas ausentes na sequência horária, sendo 56 associadas ao início do horário de verão e 48 aos dois dias sem registros do Norte;
- nenhuma carga foi preenchida e nenhuma linha foi criada artificialmente.

O dataset consolidado foi salvo em `data/processed/curva_carga_consolidado.parquet` e está pronto para a etapa de análise exploratória da série temporal.